# 02i | Drills 3: Merging, deduplicating, reshaping

Goes with **02d**: the join types of `merge`, why the row count changes after a merge, finding unmatched rows, stacking tables, deduplicating, and converting between long and wide format.

**How to use this page:** There are no input boxes here; each problem is followed directly by the answer and its output. Read the problem, work out the code in your head (or on paper), then look at the answer. After each one, cover the answer and try to recite the code.

Every problem assumes this has already been run:

```python
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()
```

<details>
<summary><b>📋 All four tables (open to compare with the results)</b></summary>

**`users`**

|   | user_id | name | country | channel | signup_date |
|---|---|---|---|---|---|
| 0 | 1 | Ann | US | organic | 2026-01-01 |
| 1 | 2 | Bob | US | ads | 2026-01-01 |
| 2 | 3 | Chen | CN | organic | 2026-01-02 |
| 3 | 4 | Dana | US | referral | 2026-01-03 |
| 4 | 5 | Eli | CA | ads | 2026-01-03 |
| 5 | 6 | Fay | CN | ads | 2026-01-05 |
| 6 | 7 | Gus | NaN | organic | 2026-01-06 |
| 7 | 8 | Hana | US | referral | 2026-01-08 |
| 8 | 9 | Ivan | CA | organic | 2026-01-09 |
| 9 | 10 | Jia | CN | ads | 2026-01-10 |

*10 rows × 5 columns*

**`orders`**

|   | order_id | user_id | order_date | amount | status |
|---|---|---|---|---|---|
| 0 | 1001 | 1 | 2026-01-02 | 30.0 | completed |
| 1 | 1002 | 1 | 2026-01-05 | 50.0 | completed |
| 2 | 1003 | 1 | 2026-01-20 | 20.0 | refunded |
| 3 | 1004 | 2 | 2026-01-03 | 15.0 | completed |
| 4 | 1005 | 2 | 2026-01-04 | 15.0 | completed |
| 5 | 1006 | 3 | 2026-01-02 | 80.0 | completed |
| 6 | 1007 | 3 | 2026-01-15 | 40.0 | completed |
| 7 | 1008 | 3 | 2026-02-01 | 60.0 | completed |
| 8 | 1009 | 4 | 2026-01-10 | 25.0 | completed |
| 9 | 1010 | 5 | 2026-01-04 | 100.0 | completed |
| 10 | 1011 | 5 | 2026-01-25 | 35.0 | refunded |
| 11 | 1012 | 6 | 2026-01-06 | 45.0 | completed |
| 12 | 1013 | 6 | 2026-02-03 | 55.0 | completed |
| 13 | 1014 | 8 | 2026-01-09 | 70.0 | completed |
| 14 | 1015 | 8 | 2026-02-05 | 70.0 | completed |
| 15 | 1016 | 10 | 2026-01-12 | 10.0 | completed |

*16 rows × 5 columns*

**`events`**

|   | user_id | event_date | event_type |
|---|---|---|---|
| 0 | 1 | 2026-01-01 | login |
| 1 | 1 | 2026-01-02 | login |
| 2 | 1 | 2026-01-02 | login |
| 3 | 1 | 2026-01-03 | login |
| 4 | 1 | 2026-01-05 | login |
| 5 | 1 | 2026-01-08 | login |
| 6 | 2 | 2026-01-01 | login |
| 7 | 2 | 2026-01-02 | login |
| 8 | 3 | 2026-01-02 | login |
| 9 | 3 | 2026-01-03 | login |
| 10 | 3 | 2026-01-04 | login |
| 11 | 3 | 2026-01-05 | login |
| 12 | 3 | 2026-01-09 | login |
| 13 | 4 | 2026-01-03 | login |
| 14 | 5 | 2026-01-03 | login |
| 15 | 5 | 2026-01-04 | login |
| 16 | 5 | 2026-01-10 | login |
| 17 | 6 | 2026-01-05 | login |
| 18 | 6 | 2026-01-06 | login |
| 19 | 6 | 2026-01-07 | login |
| 20 | 7 | 2026-01-06 | login |
| 21 | 8 | 2026-01-08 | login |
| 22 | 8 | 2026-01-09 | login |
| 23 | 8 | 2026-01-15 | login |
| 24 | 9 | 2026-01-09 | login |
| 25 | 9 | 2026-01-10 | login |
| 26 | 10 | 2026-01-10 | login |
| 27 | 1 | 2026-01-02 | view |
| 28 | 1 | 2026-01-02 | cart |
| 29 | 1 | 2026-01-02 | purchase |
| 30 | 2 | 2026-01-02 | view |
| 31 | 2 | 2026-01-02 | cart |
| 32 | 3 | 2026-01-02 | view |
| 33 | 3 | 2026-01-02 | cart |
| 34 | 3 | 2026-01-02 | purchase |
| 35 | 4 | 2026-01-03 | view |
| 36 | 5 | 2026-01-04 | view |
| 37 | 5 | 2026-01-04 | cart |
| 38 | 5 | 2026-01-04 | purchase |
| 39 | 6 | 2026-01-06 | view |
| 40 | 6 | 2026-01-06 | cart |
| 41 | 6 | 2026-01-06 | purchase |
| 42 | 7 | 2026-01-06 | view |
| 43 | 8 | 2026-01-09 | view |
| 44 | 8 | 2026-01-09 | cart |
| 45 | 8 | 2026-01-09 | purchase |
| 46 | 9 | 2026-01-09 | view |
| 47 | 9 | 2026-01-10 | cart |
| 48 | 10 | 2026-01-10 | view |

*49 rows × 3 columns*

**`employees`**

|   | emp_id | name | dept | salary | manager_id |
|---|---|---|---|---|---|
| 0 | 1 | Alice | Eng | 150 | NaN |
| 1 | 2 | Brian | Eng | 120 | 1.0 |
| 2 | 3 | Cathy | Eng | 120 | 1.0 |
| 3 | 4 | David | Eng | 95 | 2.0 |
| 4 | 5 | Emma | Sales | 90 | 1.0 |
| 5 | 6 | Frank | Sales | 70 | 5.0 |
| 6 | 7 | Grace | Sales | 70 | 5.0 |
| 7 | 8 | Henry | Mkt | 85 | 1.0 |
| 8 | 9 | Irene | Mkt | 60 | 8.0 |

*9 rows × 5 columns*

</details>

### Problem 1 · Inner join

Merge users and orders, keeping only keys present on both sides. How many rows does the result have?

```python
inner = users.merge(orders, on="user_id", how="inner")
print(len(inner))
inner[["user_id", "name", "order_id", "amount"]].head(5)
```

**Result:**

```text
16
```

|   | user_id | name | order_id | amount |
|---|---|---|---|---|
| 0 | 1 | Ann | 1001 | 30.0 |
| 1 | 1 | Ann | 1002 | 50.0 |
| 2 | 1 | Ann | 1003 | 20.0 |
| 3 | 2 | Bob | 1004 | 15.0 |
| 4 | 2 | Bob | 1005 | 15.0 |

*5 rows × 4 columns*

> `how="inner"` is the default, so you can omit it.

### Problem 2 · Left join

A left join that keeps every user. Why does it have more rows than the 10 in `users`?

```python
left = users.merge(orders, on="user_id", how="left")
print(len(users), len(left))
```

**Result:**

```text
10 18
```

> Each user's row is repeated once per order; users with no orders keep a single row.

### Problem 3 · Find unmatched rows after a left join

Use a left join to find the users who never placed an order.

```python
left = users.merge(orders, on="user_id", how="left")
left[left["order_id"].isna()][["user_id", "name"]]
```

**Result:**

|   | user_id | name |
|---|---|---|
| 13 | 7 | Gus |
| 16 | 9 | Ivan |

*2 rows × 2 columns*

> In unmatched rows, all columns from the right table are NaN.

### Problem 4 · Same thing, shorter

Names of users who never ordered, using a negated `isin`.

```python
users[~users["user_id"].isin(orders["user_id"])]["name"].tolist()
```

**Result:**

```text
['Gus', 'Ivan']
```

### Problem 5 · indicator: which side each row came from

In a left join, flag whether each user matched any order.

```python
chk = users.merge(orders[["user_id"]].drop_duplicates(), on="user_id", how="left", indicator=True)
chk[["user_id", "name", "_merge"]]
```

**Result:**

|   | user_id | name | _merge |
|---|---|---|---|
| 0 | 1 | Ann | both |
| 1 | 2 | Bob | both |
| 2 | 3 | Chen | both |
| 3 | 4 | Dana | both |
| 4 | 5 | Eli | both |
| 5 | 6 | Fay | both |
| 6 | 7 | Gus | left_only |
| 7 | 8 | Hana | both |
| 8 | 9 | Ivan | left_only |
| 9 | 10 | Jia | both |

*10 rows × 3 columns*

> Rows where `_merge` is `left_only` are the unmatched ones. The right table is deduplicated first so each user appears only once.

### Problem 6 · Add a dimension to a fact table

Add the user's country to every order.

```python
orders.merge(users[["user_id", "country"]], on="user_id", how="left").head(6)
```

**Result:**

|   | order_id | user_id | order_date | amount | status | country |
|---|---|---|---|---|---|---|
| 0 | 1001 | 1 | 2026-01-02 | 30.0 | completed | US |
| 1 | 1002 | 1 | 2026-01-05 | 50.0 | completed | US |
| 2 | 1003 | 1 | 2026-01-20 | 20.0 | refunded | US |
| 3 | 1004 | 2 | 2026-01-03 | 15.0 | completed | US |
| 4 | 1005 | 2 | 2026-01-04 | 15.0 | completed | US |
| 5 | 1006 | 3 | 2026-01-02 | 80.0 | completed | CN |

*6 rows × 6 columns*

> The orders table is on the left, so there are still 16 rows. Merging only the columns you need keeps the result clean.

### Problem 7 · Merge, then group

Total amount of completed orders per country.

```python
m = users.merge(orders[orders["status"] == "completed"], on="user_id")
m.groupby("country")["amount"].sum().reset_index().rename(columns={"amount": "total"})
```

**Result:**

|   | country | total |
|---|---|---|
| 0 | CA | 100.0 |
| 1 | CN | 290.0 |
| 2 | US | 275.0 |

*3 rows × 2 columns*

### Problem 8 · Aggregate, left join, fill with 0

Total amount of completed orders per user. All 10 users must appear, with 0 for users who have none.

```python
spend = (
    orders[orders["status"] == "completed"]
    .groupby("user_id")["amount"].sum()
    .reset_index()
    .rename(columns={"amount": "total"})
)
out = users[["user_id", "name"]].merge(spend, on="user_id", how="left")
out["total"] = out["total"].fillna(0)
out
```

**Result:**

|   | user_id | name | total |
|---|---|---|---|
| 0 | 1 | Ann | 80.0 |
| 1 | 2 | Bob | 30.0 |
| 2 | 3 | Chen | 180.0 |
| 3 | 4 | Dana | 25.0 |
| 4 | 5 | Eli | 100.0 |
| 5 | 6 | Fay | 100.0 |
| 6 | 7 | Gus | 0.0 |
| 7 | 8 | Hana | 140.0 |
| 8 | 9 | Ivan | 0.0 |
| 9 | 10 | Jia | 10.0 |

*10 rows × 3 columns*

> The most common pattern. Aggregate to "one row per user" first, so the merge still gives 10 rows.

### Problem 9 · Same pattern, with a count

Number of refunded orders per user, 0 if none.

```python
ref = (
    orders[orders["status"] == "refunded"]
    .groupby("user_id")["order_id"].count()
    .reset_index()
    .rename(columns={"order_id": "n_refund"})
)
out = users[["user_id", "name"]].merge(ref, on="user_id", how="left")
out["n_refund"] = out["n_refund"].fillna(0)
out
```

**Result:**

|   | user_id | name | n_refund |
|---|---|---|---|
| 0 | 1 | Ann | 1.0 |
| 1 | 2 | Bob | 0.0 |
| 2 | 3 | Chen | 0.0 |
| 3 | 4 | Dana | 0.0 |
| 4 | 5 | Eli | 1.0 |
| 5 | 6 | Fay | 0.0 |
| 6 | 7 | Gus | 0.0 |
| 7 | 8 | Hana | 0.0 |
| 8 | 9 | Ivan | 0.0 |
| 9 | 10 | Jia | 0.0 |

*10 rows × 3 columns*

> After filling, the values are floats like 1.0 and 0.0, because the column held NaN at one point (NaN can only live in a float column). The result is still correct.

### Problem 10 · Self-join: employees and managers

Add each employee's manager's name.

```python
mgr = employees[["emp_id", "name"]].rename(columns={"emp_id": "manager_id", "name": "manager_name"})
e = employees.merge(mgr, on="manager_id", how="left")
e[["name", "dept", "manager_id", "manager_name"]]
```

**Result:**

|   | name | dept | manager_id | manager_name |
|---|---|---|---|---|
| 0 | Alice | Eng | NaN | NaN |
| 1 | Brian | Eng | 1.0 | Alice |
| 2 | Cathy | Eng | 1.0 | Alice |
| 3 | David | Eng | 2.0 | Brian |
| 4 | Emma | Sales | 1.0 | Alice |
| 5 | Frank | Sales | 5.0 | Emma |
| 6 | Grace | Sales | 5.0 | Emma |
| 7 | Henry | Mkt | 1.0 | Alice |
| 8 | Irene | Mkt | 8.0 | Henry |

*9 rows × 4 columns*

> Merging a table with itself: make a copy to serve as the manager table, rename its columns so they line up (`emp_id` → `manager_id`), then merge. Alice has no manager, so she gets NaN.

### Problem 11 · Self-join, then group

Number of direct reports per manager.

```python
mgr = employees[["emp_id", "name"]].rename(columns={"emp_id": "manager_id", "name": "manager_name"})
e = employees.merge(mgr, on="manager_id", how="left")
e.groupby("manager_name")["emp_id"].count().reset_index().rename(columns={"emp_id": "n_reports"})
```

**Result:**

|   | manager_name | n_reports |
|---|---|---|
| 0 | Alice | 4 |
| 1 | Brian | 1 |
| 2 | Emma | 2 |
| 3 | Henry | 1 |

*4 rows × 2 columns*

### Problem 12 · Check whether the join key is unique

Before merging: what is the maximum number of rows per user in `orders`? In `users`?

```python
print(orders["user_id"].value_counts().max())
print(users["user_id"].value_counts().max())
```

**Result:**

```text
3
1
```

> Each id appears once in `users` and up to 3 times in `orders`. So when `users` is left-joined to `orders`, some users are repeated into 3 rows.

### Problem 13 · Stack tables vertically

Split the orders into a January table and a February table, then stack them back together.

```python
jan = orders[orders["order_date"] < "2026-02-01"]
feb = orders[orders["order_date"] >= "2026-02-01"]
both = pd.concat([jan, feb]).reset_index(drop=True)
print(len(jan), len(feb), len(both))
both.head(4)
```

**Result:**

```text
13 3 16
```

|   | order_id | user_id | order_date | amount | status |
|---|---|---|---|---|---|
| 0 | 1001 | 1 | 2026-01-02 | 30.0 | completed |
| 1 | 1002 | 1 | 2026-01-05 | 50.0 | completed |
| 2 | 1003 | 1 | 2026-01-20 | 20.0 | refunded |
| 3 | 1004 | 2 | 2026-01-03 | 15.0 | completed |

*4 rows × 5 columns*

> R: `bind_rows`. `reset_index(drop=True)` renumbers the rows.

### Problem 14 · Drop fully duplicated rows

Are there fully duplicated rows in the login records? How many rows before and after deduplication?

```python
logins = events[events["event_type"] == "login"]
print(len(logins), len(logins.drop_duplicates()))
```

**Result:**

```text
27 26
```

> User 1 has two identical login records on January 2.

### Problem 15 · Deduplicate on some columns, keep the first

The row for each user's first login.

```python
(
    events[events["event_type"] == "login"]
    .sort_values("event_date")
    .drop_duplicates(subset=["user_id"], keep="first")
)
```

**Result:**

|   | user_id | event_date | event_type |
|---|---|---|---|
| 0 | 1 | 2026-01-01 | login |
| 6 | 2 | 2026-01-01 | login |
| 8 | 3 | 2026-01-02 | login |
| 13 | 4 | 2026-01-03 | login |
| 14 | 5 | 2026-01-03 | login |
| 17 | 6 | 2026-01-05 | login |
| 20 | 7 | 2026-01-06 | login |
| 21 | 8 | 2026-01-08 | login |
| 24 | 9 | 2026-01-09 | login |
| 26 | 10 | 2026-01-10 | login |

*10 rows × 3 columns*

> Sort in the order you want first, then deduplicate on the grouping column.

### Problem 16 · Keep the last row

Each user's most recent order.

```python
(
    orders.sort_values("order_date")
    .drop_duplicates(subset=["user_id"], keep="last")
    [["user_id", "order_id", "order_date"]]
    .sort_values("user_id")
)
```

**Result:**

|   | user_id | order_id | order_date |
|---|---|---|---|
| 2 | 1 | 1003 | 2026-01-20 |
| 4 | 2 | 1005 | 2026-01-04 |
| 7 | 3 | 1008 | 2026-02-01 |
| 8 | 4 | 1009 | 2026-01-10 |
| 10 | 5 | 1011 | 2026-01-25 |
| 12 | 6 | 1013 | 2026-02-03 |
| 14 | 8 | 1015 | 2026-02-05 |
| 15 | 10 | 1016 | 2026-01-12 |

*8 rows × 3 columns*

### Problem 17 · Wide to long: melt

Turn a score table with one row per student into one row per student per subject.

```python
wide = pd.DataFrame({"name": ["Ann", "Bob", "Cy"], "math": [90, 70, 85], "stats": [85, 95, 60]})
wide.melt(id_vars="name", var_name="subject", value_name="score")
```

**Result:**

|   | name | subject | score |
|---|---|---|---|
| 0 | Ann | math | 90 |
| 1 | Bob | math | 70 |
| 2 | Cy | math | 85 |
| 3 | Ann | stats | 85 |
| 4 | Bob | stats | 95 |
| 5 | Cy | stats | 60 |

*6 rows × 3 columns*

> R: `pivot_longer`. `id_vars` are the columns that stay fixed.

### Problem 18 · Long to wide: pivot

Turn the long table from the previous problem back into a wide one.

```python
wide = pd.DataFrame({"name": ["Ann", "Bob", "Cy"], "math": [90, 70, 85], "stats": [85, 95, 60]})
long = wide.melt(id_vars="name", var_name="subject", value_name="score")
long.pivot(index="name", columns="subject", values="score").reset_index()
```

**Result:**

|   | name | math | stats |
|---|---|---|---|
| 0 | Ann | 90 | 85 |
| 1 | Bob | 70 | 95 |
| 2 | Cy | 85 | 60 |

*3 rows × 3 columns*

### Problem 19 · Long to wide with counts: pivot_table

Number of events of each type per user, 0 if none.

```python
events.pivot_table(index="user_id", columns="event_type", values="event_date", aggfunc="count", fill_value=0)
```

**Result:**

| user_id | cart | login | purchase | view |
|---|---|---|---|---|
| 1 | 1 | 6 | 1 | 1 |
| 2 | 1 | 2 | 0 | 1 |
| 3 | 1 | 5 | 1 | 1 |
| 4 | 0 | 1 | 0 | 1 |
| 5 | 1 | 3 | 1 | 1 |
| 6 | 1 | 3 | 1 | 1 |
| 7 | 0 | 1 | 0 | 1 |
| 8 | 1 | 3 | 1 | 1 |
| 9 | 1 | 2 | 0 | 1 |
| 10 | 0 | 1 | 0 | 1 |

*10 rows × 4 columns*

> Any column works for `values`; `aggfunc="count"` counts rows.

### Problem 20 · In A but not in B

Users who viewed a product (`view`) but never bought anything (`purchase`).

```python
viewers = events[events["event_type"] == "view"]["user_id"]
buyers = events[events["event_type"] == "purchase"]["user_id"]
users[users["user_id"].isin(viewers) & ~users["user_id"].isin(buyers)][["user_id", "name"]]
```

**Result:**

|   | user_id | name |
|---|---|---|
| 1 | 2 | Bob |
| 3 | 4 | Dana |
| 6 | 7 | Gus |
| 8 | 9 | Ivan |
| 9 | 10 | Jia |

*5 rows × 2 columns*

## Recap

| Task | Code |
|---|---|
| Merge | `a.merge(b, on="key", how="left")` |
| Key names differ | `rename` them to match, then merge |
| Unmatched rows | `~a["key"].isin(b["key"])`, or `indicator=True` |
| Merge without multiplying rows | First aggregate the "many" side to one row per key |
| Stack | `pd.concat([df1, df2]).reset_index(drop=True)` |
| Deduplicate | `drop_duplicates(subset=[...], keep="first")` |
| Long / wide | `melt` / `pivot` / `pivot_table` |